# Практична робота №1
## Розгортання середовища та знайомство з IoT-даними

У цій роботі ви:

- перевірите роботу контейнеризованого середовища;
- ознайомитеся зі структурою навчального проєкту;
- прочитаєте файли JSON, CSV і JSON Lines;
- визначите склад IoT-системи;
- виконаєте базові підрахунки без очищення даних;
- збережете результати у визначеному форматі.

> Виконуйте комірки послідовно зверху вниз. Не змінюйте файли в каталозі `data/input`.

## Дані студента

**Прізвище, ім’я та по батькові:** _вкажіть тут_  
**Група:** _вкажіть тут_  
**Дата виконання:** _вкажіть тут_  

Ідентифікатор варіанта та студентський ідентифікатор далі будуть прочитані з `metadata.json`.

## Межі цієї роботи

У практичній роботі №1 **не потрібно**:

- очищувати або виправляти події;
- шукати дублікати;
- перевіряти допустимі межі значень;
- перевіряти правильність часових міток;
- виявляти невідомі пристрої чи непідтримувані метрики;
- змінювати початкові файли.

Ці завдання виконуватимуться в наступній практичній роботі з якості даних.

## 1. Перевірка робочого середовища

In [6]:
import json
import sys
from pathlib import Path

import duckdb
import polars as pl
import pyarrow as pa

print("Python:", sys.version.split()[0])
print("Polars:", pl.__version__)
print("DuckDB:", duckdb.__version__)
print("PyArrow:", pa.__version__)

Python: 3.12.13
Polars: 1.41.2
DuckDB: 1.5.4
PyArrow: 24.0.0


In [7]:
PROJECT_DIR = Path("/workspace")
INPUT_DIR = PROJECT_DIR / "data" / "input"
WORKING_DIR = PROJECT_DIR / "data" / "working"
RESULTS_DIR = PROJECT_DIR / "results" / "practical_01"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("INPUT_DIR:", INPUT_DIR)
print("WORKING_DIR:", WORKING_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

PROJECT_DIR: /workspace
INPUT_DIR: /workspace/data/input
WORKING_DIR: /workspace/data/working
RESULTS_DIR: /workspace/results/practical_01


In [8]:
# Повна автоматична перевірка середовища.
!python /workspace/scripts/check_environment.py

=== Перевірка робочого середовища ===
Python:  3.12.13
Polars:  1.41.2
DuckDB:  1.5.4
PyArrow: 24.0.0
[ OK ] версії Python-бібліотек
[ OK ] структура каталогів
[ OK ] наявність п'яти вхідних файлів
       Варіант: 1 | Студент: Іваненко Іван Іванович
[ OK ] структура metadata.json
[ OK ] читання першої JSONL-події
       Пристроїв: 38 | Метрик: 12 | Пар тип-метрика: 12
[ OK ] читання CSV-довідників через Polars
[ OK ] data/input змонтовано лише для читання
[ OK ] data/working доступний для запису
[ OK ] results доступний для запису

Середовище готове до роботи.


### Результат перевірки

Запишіть, чи завершилася перевірка повідомленням **«Середовище готове до роботи»**.

**Відповідь:** _вкажіть тут_

## 2. Структура навчального проєкту

### Завдання 2.1

Програмно виведіть вміст кореневого каталогу `/workspace`. Для кожного об’єкта покажіть його назву та тип: файл або каталог.

In [16]:
# TODO: виведіть об'єкти, що розташовані безпосередньо в PROJECT_DIR.
# Підказка: використайте PROJECT_DIR.iterdir(), path.name та path.is_dir().
for path in sorted(PROJECT_DIR.iterdir(), key=lambda item: item.name):
    object_type = "каталог" if path.is_dir() else "файл"
    print(f"{path.name:<20} — {object_type}")

data                 — каталог
notebooks            — каталог
results              — каталог
scripts              — каталог
src                  — каталог


### Завдання 2.2

Коротко поясніть призначення каталогів:

- `data/input`;
- `data/working`;
- `notebooks`;
- `results`;
- `src`.

**Відповідь:**

_впишіть пояснення тут_

## 3. Перевірка комплектності індивідуального пакета

Перш ніж аналізувати дані, необхідно переконатися, що всі файли варіанта наявні та не порожні.


In [10]:
REQUIRED_FILES = [
    "raw_iot_events.jsonl",
    "device_registry.csv",
    "metric_catalog.csv",
    "device_type_metrics.csv",
    "metadata.json",
]

# Приклад роботи з одним шляхом.
example_path = INPUT_DIR / "metadata.json"

print("Шлях:", example_path)
print("Файл існує:", example_path.exists())
print(
    "Розмір у байтах:",
    example_path.stat().st_size if example_path.exists() else None,
)


Шлях: /workspace/data/input/metadata.json
Файл існує: True
Розмір у байтах: 351


### Завдання 3.1

Для кожного файла зі списку `REQUIRED_FILES` визначте:

- назву файла;
- чи існує він;
- розмір у байтах;
- чи є він порожнім.

Сформуйте Polars DataFrame `file_inventory` зі стовпцями:

| filename | exists | size_bytes | is_empty |
|---|---:|---:|---:|

Використайте цикл, `Path.exists()` і `Path.stat().st_size`.


In [17]:
inventory_rows = []

for filename in REQUIRED_FILES:
    file_path = INPUT_DIR / filename

    exists = file_path.exists()

    size_bytes = (
        file_path.stat().st_size
        if exists
        else None
    )

    is_empty = (
        size_bytes == 0
        if exists
        else None
    )


    inventory_rows.append(
        {
            "filename": filename,
            "exists": exists,
            "size_bytes": size_bytes,
            "is_empty": is_empty,
        }
    )

file_inventory = pl.DataFrame(inventory_rows)
file_inventory


filename,exists,size_bytes,is_empty
str,bool,i64,bool
"""raw_iot_events.jsonl""",true,47842679,false
"""device_registry.csv""",true,1879,false
"""metric_catalog.csv""",true,339,false
"""device_type_metrics.csv""",true,318,false
"""metadata.json""",true,351,false


In [18]:
# Самоперевірка завдання 3.1.
EXPECTED_INVENTORY_COLUMNS = [
    "filename",
    "exists",
    "size_bytes",
    "is_empty",
]

assert isinstance(file_inventory, pl.DataFrame), (
    "file_inventory повинен бути Polars DataFrame"
)
assert file_inventory.columns == EXPECTED_INVENTORY_COLUMNS, (
    f"Очікувані стовпці: {EXPECTED_INVENTORY_COLUMNS}"
)
assert file_inventory.height == len(REQUIRED_FILES), (
    "У таблиці має бути один рядок для кожного обов'язкового файла"
)
assert file_inventory["exists"].all(), (
    "Не всі обов'язкові файли знайдено"
)
assert (file_inventory["size_bytes"] > 0).all(), (
    "Один або кілька вхідних файлів порожні"
)
assert not file_inventory["is_empty"].any(), (
    "Один або кілька вхідних файлів позначено як порожні"
)

print("[ OK ] Усі обов'язкові файли знайдено і вони не порожні.")


[ OK ] Усі обов'язкові файли знайдено і вони не порожні.


### Висновок до розділу

Вкажіть:

1. Чи всі необхідні файли наявні?
2. Який файл має найбільший розмір?
3. Чому файл із подіями значно більший за довідники?

**Відповідь:**

_впишіть відповідь тут_


## 4. Метадані індивідуального варіанта

Файл `metadata.json` описує не окремі події, а весь виданий студенту набір даних.


### Завдання 4.1

Прочитайте `metadata.json` за допомогою стандартного модуля `json`.

Результат збережіть у змінній `metadata`. Вона повинна містити словник Python.


In [19]:
metadata_path = INPUT_DIR / "metadata.json"

# TODO: відкрийте файл у режимі читання з кодуванням UTF-8
# та прочитайте JSON за допомогою json.load().
with metadata_path.open("r", encoding="utf-8") as file:
    metadata = json.load(file)

metadata


{'course': 'Основи IoT та аналітики великих даних',
 'dataset_version': '2026.1',
 'variant_id': 1,
 'group': 'КБ-31',
 'student_id': 'KB31-01',
 'student_name': 'Іваненко Іван Іванович',
 'period_start': '2026-03-01T00:00:00Z',
 'period_end': '2026-03-07T23:59:59Z',
 'timezone': 'UTC'}

In [20]:
# Самоперевірка завдання 4.1.
EXPECTED_METADATA_FIELDS = {
    "course",
    "dataset_version",
    "variant_id",
    "group",
    "student_id",
    "student_name",
    "period_start",
    "period_end",
    "timezone",
}

assert isinstance(metadata, dict), (
    "metadata повинен бути словником Python"
)

missing_fields = EXPECTED_METADATA_FIELDS - set(metadata)
assert not missing_fields, (
    f"У metadata відсутні поля: {sorted(missing_fields)}"
)

print("[ OK ] metadata.json успішно прочитано.")
print("Кількість полів:", len(metadata))


[ OK ] metadata.json успішно прочитано.
Кількість полів: 9


### Завдання 4.2

Створіть словник `variant_summary`, вибравши з `metadata` такі поля:

- `course`;
- `dataset_version`;
- `variant_id`;
- `group`;
- `student_id`;
- `student_name`;
- `period_start`;
- `period_end`;
- `timezone`.

Не вписуйте значення вручну — отримайте їх зі словника `metadata`.


In [21]:
variant_summary = {
    "course": metadata["course"],
    "dataset_version": metadata["dataset_version"],
    "variant_id": metadata["variant_id"],
    "group": metadata["group"],
    "student_id": metadata["student_id"],
    "student_name": metadata["student_name"],
    "period_start": metadata["period_start"],
    "period_end": metadata["period_end"],
    "timezone": metadata["timezone"],
}

variant_summary


{'course': 'Основи IoT та аналітики великих даних',
 'dataset_version': '2026.1',
 'variant_id': 1,
 'group': 'КБ-31',
 'student_id': 'KB31-01',
 'student_name': 'Іваненко Іван Іванович',
 'period_start': '2026-03-01T00:00:00Z',
 'period_end': '2026-03-07T23:59:59Z',
 'timezone': 'UTC'}

In [22]:
# Самоперевірка завдання 4.2.
assert set(variant_summary) == EXPECTED_METADATA_FIELDS, (
    "variant_summary повинен містити всі дев'ять потрібних полів"
)
assert variant_summary["variant_id"] == metadata["variant_id"]
assert variant_summary["student_id"] == metadata["student_id"]

print("[ OK ] Короткий опис варіанта сформовано.")
print(
    f"Варіант: {variant_summary['variant_id']} | "
    f"Студент: {variant_summary['student_name']}"
)


[ OK ] Короткий опис варіанта сформовано.
Варіант: 1 | Студент: Іваненко Іван Іванович


### Висновок до розділу

Коротко поясніть:

1. Чим `metadata.json` відрізняється від файла подій?
2. Який часовий період заявлено для вашого варіанта?
3. Для чого в метаданих явно вказана часова зона?

**Відповідь:**

_впишіть відповідь тут_


## 5. Реєстр IoT-пристроїв

Файл `device_registry.csv` описує зареєстровані в системі пристрої. Один рядок відповідає одному пристрою.

### Завдання 5.1

Завантажте файл у Polars DataFrame `device_registry`. Виведіть:

- перші 5 рядків;
- розмір таблиці;
- назви та типи стовпців.

In [23]:
device_registry = pl.read_csv(
    INPUT_DIR / "device_registry.csv"
)

print("Перші 5 рядків:")
display(device_registry.head())

print("Розмір таблиці:", device_registry.shape)
print("Схема:")
print(device_registry.schema)

Перші 5 рядків:


device_id,device_type,location,criticality,expected_interval_sec
str,str,str,str,i64
"""dev-001""","""smart_meter""","""building-1""","""high""",120
"""dev-002""","""access_controller""","""network-zone-A""","""medium""",60
"""dev-003""","""access_controller""","""building-1""","""medium""",60
"""dev-004""","""climate_sensor""","""office-main""","""medium""",300
"""dev-005""","""climate_sensor""","""office-main""","""medium""",300


Розмір таблиці: (38, 5)
Схема:
Schema({'device_id': String, 'device_type': String, 'location': String, 'criticality': String, 'expected_interval_sec': Int64})


### Завдання 5.2

Визначте:

- кількість зареєстрованих пристроїв;
- кількість типів пристроїв;
- кількість локацій;
- перелік рівнів критичності;
- перелік очікуваних інтервалів передавання даних.

In [24]:
registered_devices = device_registry.height
device_types_count = device_registry["device_type"].n_unique()
locations_count = device_registry["location"].n_unique()

criticality_levels = (
    device_registry["criticality"]
    .unique()
    .sort()
    .to_list()
)

expected_intervals = (
    device_registry["expected_interval_sec"]
    .unique()
    .sort()
    .to_list()
)

print("Зареєстрованих пристроїв:", registered_devices)
print("Типів пристроїв:", device_types_count)
print("Локацій:", locations_count)
print("Рівні критичності:", criticality_levels)
print("Інтервали передавання:", expected_intervals)

Зареєстрованих пристроїв: 38
Типів пристроїв: 5
Локацій: 5
Рівні критичності: ['critical', 'high', 'medium']
Інтервали передавання: [30, 60, 120, 300]


### Завдання 5.3

Побудуйте дві таблиці:

1. `devices_by_type` зі стовпцями `device_type`, `device_count`;
2. `devices_by_location` зі стовпцями `location`, `device_count`.

Відсортуйте обидві таблиці за спаданням кількості пристроїв.

In [25]:
devices_by_type = (
    device_registry
    .group_by("device_type")
    .agg(
        pl.len().alias("device_count")
    )
    .sort(
        ["device_count", "device_type"],
        descending=[True, False]
    )
)

devices_by_location = (
    device_registry
    .group_by("location")
    .agg(
        pl.len().alias("device_count")
    )
    .sort(
        ["device_count", "location"],
        descending=[True, False]
    )
)

display(devices_by_type)
display(devices_by_location)

device_type,device_count
str,u32
"""climate_sensor""",11
"""industrial_sensor""",8
"""access_controller""",7
"""network_gateway""",6
"""smart_meter""",6


location,device_count
str,u32
"""network-zone-A""",12
"""office-main""",9
"""workshop-2""",9
"""building-1""",6
"""greenhouse-A""",2


### Висновок до розділу

Опишіть склад системи: які типи пристроїв і локації в ній представлені? Що можуть означати поля `criticality` та `expected_interval_sec`?

**Відповідь:** _вкажіть тут_

## 6. Каталог метрик і зв’язки з типами пристроїв

### Завдання 6.1

Завантажте:

- `metric_catalog.csv` у `metric_catalog`;
- `device_type_metrics.csv` у `device_type_metrics`.

Для кожної таблиці виведіть перші рядки, розмір і схему.

In [26]:
metric_catalog = pl.read_csv(
    INPUT_DIR / "metric_catalog.csv"
)

device_type_metrics = pl.read_csv(
    INPUT_DIR / "device_type_metrics.csv"
)

print("Каталог метрик:")
display(metric_catalog.head())
print("Розмір:", metric_catalog.shape)
print("Схема:", metric_catalog.schema)

print("\nМетрики за типами пристроїв:")
display(device_type_metrics.head())
print("Розмір:", device_type_metrics.shape)
print("Схема:", device_type_metrics.schema)

Каталог метрик:


metric,unit,min_value,max_value,value_type
str,str,i64,i64,str
"""temperature""","""C""",-20,80,"""float"""
"""humidity""","""%""",0,100,"""float"""
"""co2""","""ppm""",300,5000,"""float"""
"""power_w""","""W""",0,10000,"""float"""
"""voltage""","""V""",180,260,"""float"""


Розмір: (12, 5)
Схема: Schema({'metric': String, 'unit': String, 'min_value': Int64, 'max_value': Int64, 'value_type': String})

Метрики за типами пристроїв:


device_type,metric
str,str
"""climate_sensor""","""temperature"""
"""climate_sensor""","""humidity"""
"""climate_sensor""","""co2"""
"""smart_meter""","""power_w"""
"""smart_meter""","""voltage"""


Розмір: (12, 2)
Схема: Schema({'device_type': String, 'metric': String})


### Завдання 6.2

Визначте:

- кількість метрик у каталозі;
- перелік одиниць вимірювання;
- перелік очікуваних типів значень;
- кількість дозволених пар `device_type–metric`.

In [27]:
catalog_metrics_count = metric_catalog.height

measurement_units = (
    metric_catalog["unit"]
    .unique()
    .sort()
    .to_list()
)

value_types = (
    metric_catalog["value_type"]
    .unique()
    .sort()
    .to_list()
)

allowed_pairs_count = (
    device_type_metrics
    .unique(subset=["device_type", "metric"])
    .height
)

print("Метрик у каталозі:", catalog_metrics_count)
print("Одиниці вимірювання:", measurement_units)
print("Типи значень:", value_types)
print("Дозволених пар тип–метрика:", allowed_pairs_count)

Метрик у каталозі: 12
Одиниці вимірювання: ['%', 'A', 'C', 'V', 'W', 'bool', 'mm/s', 'ms', 'ppm']
Типи значень: ['float', 'int']
Дозволених пар тип–метрика: 12


### Завдання 6.3

Створіть таблицю `metrics_by_device_type` зі стовпцями:

| device_type | metric_count |
|---|---:|

Вона повинна показувати, скільки основних телеметричних метрик підтримує кожен тип пристрою.

In [28]:
metrics_by_device_type = (
    device_type_metrics
    .group_by("device_type")
    .agg(
        pl.len().alias("metric_count")
    )
    .sort(
        ["metric_count", "device_type"],
        descending=[True, False]
    )
)

metrics_by_device_type

device_type,metric_count
str,u32
"""climate_sensor""",3
"""smart_meter""",3
"""access_controller""",2
"""industrial_sensor""",2
"""network_gateway""",2


### Завдання 6.4

Для кожного типу пристрою виведіть перелік підтримуваних ним метрик.

In [29]:
metric_lists_by_device_type = (
    device_type_metrics
    .group_by("device_type")
    .agg(
        pl.col("metric")
        .sort()
        .alias("metrics")
    )
    .sort("device_type")
)

metric_lists_by_device_type

device_type,metrics
str,list[str]
"""access_controller""","[""door_state"", ""motion""]"
"""climate_sensor""","[""co2"", ""humidity"", ""temperature""]"
"""industrial_sensor""","[""temperature"", ""vibration""]"
"""network_gateway""","[""latency_ms"", ""packet_loss""]"
"""smart_meter""","[""current_a"", ""power_w"", ""voltage""]"


### Висновок до розділу

Поясніть зв’язок між `device_registry.csv`, `metric_catalog.csv` і `device_type_metrics.csv`.

**Відповідь:** _вкажіть тут_

## 7. Формат JSON Lines і структура події

Файл `raw_iot_events.jsonl` містить необроблені події. У форматі JSON Lines кожен рядок є окремим JSON-об’єктом.

### Завдання 7.1

Прочитайте перші 5 рядків як звичайний текст. Не завантажуйте поки весь файл.

In [30]:
events_path = INPUT_DIR / "raw_iot_events.jsonl"

first_five_lines = []

with events_path.open("r", encoding="utf-8") as file:
    for _ in range(5):
        line = file.readline()

        if not line:
            break

        first_five_lines.append(line.strip())

first_five_lines

['{"event_id":"evt-v01-00086689","event_ts":"2026-03-01T00:00:00Z","device_id":"dev-012","event_type":"telemetry","metric":"co2","value":683.57}',
 '{"event_id":"evt-v01-00088705","event_ts":"2026-03-01T00:00:00Z","device_id":"dev-013","event_type":"telemetry","metric":"vibration","value":2.84}',
 '{"event_id":"evt-v01-00098785","event_ts":"2026-03-01T00:00:00Z","device_id":"dev-014","event_type":"network","metric":"packet_loss","value":1.59}',
 '{"event_id":"evt-v01-00183457","event_ts":"2026-03-01T00:00:00Z","device_id":"dev-024","event_type":"status","metric":"battery_level","value":88.63}',
 '{"event_id":"evt-v01-00240913","event_ts":"2026-03-01T00:00:00Z","device_id":"dev-029","event_type":"telemetry","metric":"packet_loss","value":1.17}']

### Завдання 7.2

Перетворіть перший рядок із JSON-тексту на словник Python та виведіть:

- сам словник;
- перелік полів;
- Python-тип значення кожного поля.

In [31]:
first_event = json.loads(first_five_lines[0])

print("Подія:")
print(
    json.dumps(
        first_event,
        ensure_ascii=False,
        indent=2
    )
)

print("\nПоля:")
print(list(first_event.keys()))

print("\nТипи значень:")

for field, value in first_event.items():
    print(
        f"{field:<12} → "
        f"{type(value).__name__}"
    )

Подія:
{
  "event_id": "evt-v01-00086689",
  "event_ts": "2026-03-01T00:00:00Z",
  "device_id": "dev-012",
  "event_type": "telemetry",
  "metric": "co2",
  "value": 683.57
}

Поля:
['event_id', 'event_ts', 'device_id', 'event_type', 'metric', 'value']

Типи значень:
event_id     → str
event_ts     → str
device_id    → str
event_type   → str
metric       → str
value        → float


### Завдання 7.3

Поясніть семантику полів:

- `event_id`;
- `event_ts`;
- `device_id`;
- `event_type`;
- `metric`;
- `value`.

Також поясніть, чим JSON Lines відрізняється від одного звичайного JSON-масиву.

**Відповідь:** _вкажіть тут_

## 8. Завантаження набору подій

### Завдання 8.1

Завантажте весь файл `raw_iot_events.jsonl` у Polars DataFrame `events` за допомогою `pl.read_ndjson()`.

Після завантаження виведіть:

- перші 5 рядків;
- кількість рядків і стовпців;
- назви та типи стовпців.

> На цьому етапі не виправляйте жодних значень і не видаляйте рядки.

In [32]:
events = pl.read_ndjson(events_path)

print("Перші 5 подій:")
display(events.head())

print("Розмір таблиці:", events.shape)
print("Схема:")
print(events.schema)

Перші 5 подій:


event_id,event_ts,device_id,event_type,metric,value
str,str,str,str,str,f64
"""evt-v01-00086689""","""2026-03-01T00:00:00Z""","""dev-012""","""telemetry""","""co2""",683.57
"""evt-v01-00088705""","""2026-03-01T00:00:00Z""","""dev-013""","""telemetry""","""vibration""",2.84
"""evt-v01-00098785""","""2026-03-01T00:00:00Z""","""dev-014""","""network""","""packet_loss""",1.59
"""evt-v01-00183457""","""2026-03-01T00:00:00Z""","""dev-024""","""status""","""battery_level""",88.63
"""evt-v01-00240913""","""2026-03-01T00:00:00Z""","""dev-029""","""telemetry""","""packet_loss""",1.17


Розмір таблиці: (324561, 6)
Схема:
Schema({'event_id': String, 'event_ts': String, 'device_id': String, 'event_type': String, 'metric': String, 'value': Float64})


### Завдання 8.2

Визначте лише базові характеристики:

- загальну кількість рядків;
- кількість різних значень `device_id`;
- кількість різних метрик;
- перелік типів подій;
- кількість подій кожного типу.

Не досліджуйте причини можливих розбіжностей із довідниками.

In [33]:
raw_event_rows = events.height

observed_device_ids = (
    events["device_id"]
    .drop_nulls()
    .n_unique()
)

observed_metrics = (
    events["metric"]
    .drop_nulls()
    .n_unique()
)

event_types = (
    events["event_type"]
    .drop_nulls()
    .unique()
    .sort()
    .to_list()
)

events_by_type = (
    events
    .filter(
        pl.col("event_type").is_not_null()
    )
    .group_by("event_type")
    .agg(
        pl.len().alias("event_count")
    )
    .sort("event_count", descending=True)
)

missing_event_type_rows = (
    events["event_type"].null_count()
)

print("Усього рядків:", raw_event_rows)
print("Унікальних непорожніх device_id:", observed_device_ids)
print("Унікальних непорожніх метрик:", observed_metrics)
print("Типи подій:", event_types)
print("Рядків без event_type:", missing_event_type_rows)

events_by_type

Усього рядків: 324561
Унікальних непорожніх device_id: 61
Унікальних непорожніх метрик: 12
Типи подій: ['network', 'status', 'telemetry']
Рядків без event_type: 16


event_type,event_count
str,u32
"""telemetry""",243268
"""status""",48873
"""network""",32404


### Висновок до розділу

Які типи подій представлені в наборі? Чим, на вашу думку, відрізняються `telemetry`, `status` і `network`?

**Відповідь:** _вкажіть тут_

## 9. Узагальнена модель IoT-системи

На основі всіх п’яти файлів опишіть шлях даних у системі:

```text
device_type → device → event → metric → value
```

У поясненні вкажіть:

1. де зберігається інформація про конкретний пристрій;
2. де описані метрики та їх одиниці вимірювання;
3. де задано, які телеметричні метрики підтримує тип пристрою;
4. де зберігаються фактичні події;
5. як за `device_id` подію можна пов’язати з типом і локацією пристрою.

**Відповідь:** _вкажіть тут_

## 10. Збереження результатів

У каталозі `results/practical_01` потрібно створити:

```text
system_overview.json
devices_by_type.csv
devices_by_location.csv
metrics_by_device_type.csv
events_by_type.csv
```

Файл `system_overview.json` повинен мати таку структуру:

```json
{
  "dataset_version": "...",
  "variant_id": 0,
  "student_id": "...",
  "registered_devices": 0,
  "device_types": 0,
  "locations": 0,
  "catalog_metrics": 0,
  "allowed_device_type_metric_pairs": 0,
  "raw_event_rows": 0,
  "observed_device_ids": 0,
  "observed_metrics": 0,
  "event_types": []
}
```

Усі значення мають бути отримані програмно з вхідних файлів.

In [34]:
system_overview = {
    "dataset_version": metadata["dataset_version"],
    "variant_id": metadata["variant_id"],
    "student_id": metadata["student_id"],
    "registered_devices": registered_devices,
    "device_types": device_types_count,
    "locations": locations_count,
    "catalog_metrics": catalog_metrics_count,
    "allowed_device_type_metric_pairs": allowed_pairs_count,
    "raw_event_rows": raw_event_rows,
    "observed_device_ids": observed_device_ids,
    "observed_metrics": observed_metrics,
    "event_types": event_types,
}

system_overview

{'dataset_version': '2026.1',
 'variant_id': 1,
 'student_id': 'KB31-01',
 'registered_devices': 38,
 'device_types': 5,
 'locations': 5,
 'catalog_metrics': 12,
 'allowed_device_type_metric_pairs': 12,
 'raw_event_rows': 324561,
 'observed_device_ids': 61,
 'observed_metrics': 12,
 'event_types': ['network', 'status', 'telemetry']}

### Самоперевірка вихідних файлів

In [35]:
EXPECTED_RESULT_FILES = [
    "system_overview.json",
    "devices_by_type.csv",
    "devices_by_location.csv",
    "metrics_by_device_type.csv",
    "events_by_type.csv",
]

EXPECTED_OVERVIEW_KEYS = {
    "dataset_version",
    "variant_id",
    "student_id",
    "registered_devices",
    "device_types",
    "locations",
    "catalog_metrics",
    "allowed_device_type_metric_pairs",
    "raw_event_rows",
    "observed_device_ids",
    "observed_metrics",
    "event_types",
}

missing_result_files = [
    name for name in EXPECTED_RESULT_FILES
    if not (RESULTS_DIR / name).is_file()
]

if missing_result_files:
    print("Відсутні файли:", missing_result_files)
else:
    print("Усі очікувані файли створено.")

    saved_overview = json.loads(
        (RESULTS_DIR / "system_overview.json").read_text(encoding="utf-8")
    )
    missing_keys = sorted(EXPECTED_OVERVIEW_KEYS - set(saved_overview))

    if missing_keys:
        print("У system_overview.json відсутні поля:", missing_keys)
    else:
        print("Структура system_overview.json правильна.")

Відсутні файли: ['system_overview.json', 'devices_by_type.csv', 'devices_by_location.csv', 'metrics_by_device_type.csv', 'events_by_type.csv']


## 11. Підсумкові висновки

Сформулюйте висновок обсягом приблизно 8–12 речень. Обов’язково зазначте:

- чи вдалося розгорнути й перевірити середовище;
- з яких файлів складається індивідуальний пакет;
- які сутності описують ці файли;
- які типи пристроїв, метрик і подій представлені у вашому варіанті;
- чому вхідні дані відокремлені від робочих і вихідних файлів;
- що нового ви навчилися робити в Python, Polars і JupyterLab.

**Висновок:**

_впишіть текст тут_

## 12. Контрольний список перед здачею

- [ ] Заповнено дані студента.
- [ ] Усі комірки виконано послідовно.
- [ ] У ноутбуці збережено результати виконання комірок.
- [ ] Усі текстові відповіді та висновки заповнено.
- [ ] Початкові файли в `data/input` не змінено.
- [ ] У `results/practical_01` створено п’ять необхідних файлів.
- [ ] Комірка самоперевірки не повідомляє про відсутні файли або поля.